# Notebook 05 — Perfil Agregado por Parlamentar

**Sprint 3 — Lei e Política**

Perfil descritivo (não-supervisionado) de cada deputado por tema, consumido pelo frontend
da Sprint 4. Para cada par (parlamentar × tema) calculamos o percentual de votos
favoráveis e classificamos a postura.

## Pipeline

1. Junção `votos` × `votacoes` × `proposicoes` (tema_cidadao) × `parlamentares`
2. Agregação: `pct_favoravel` e `total_votacoes` por parlamentar × tema (mín. 3 votos decisivos)
3. Classificação de `postura_geral`: ≥65% → favorável · ≤35% → contrário · senão neutro
4. Gravação em `perfil_parlamentar`

In [1]:
import sys
sys.path.insert(0, '..')

import logging

import pandas as pd

from src.db import buscar_todos, upsert_perfil

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('05_perfil')
print('Módulos carregados.')

Módulos carregados.


## 1. Montagem da base de votos

Cada linha é um voto decisivo (`favoravel`/`contrario`) de um deputado, ligado ao tema da
proposição. Juntamos `votos` → `votacoes` → `proposicoes` (`tema_cidadao`) →
`parlamentares`. Votos sem tema (votação sem proposição linkada) e abstenções saem do
cálculo de percentual.

In [2]:
# Carrega e junta as tabelas (escopo: Câmara — só há votos nominais da Câmara)
votos = pd.DataFrame(buscar_todos('votos', 'votacao_id,parlamentar_id,voto'))
votacoes = pd.DataFrame(buscar_todos('votacoes', 'id,proposicao_id'))
proposicoes = pd.DataFrame(buscar_todos('proposicoes', 'id,tema_cidadao'))
parlamentares = pd.DataFrame(buscar_todos('parlamentares', 'id,nome,casa'))

print(f'votos={len(votos)}  votacoes={len(votacoes)}  '
      f'proposicoes={len(proposicoes)}  parlamentares={len(parlamentares)}')

# Chaves de junção como inteiro anulável (proposicao_id vem como object por conter nulos)
for d, col in [(votos, 'votacao_id'), (votos, 'parlamentar_id'),
               (votacoes, 'id'), (votacoes, 'proposicao_id'),
               (proposicoes, 'id'), (parlamentares, 'id')]:
    d[col] = pd.to_numeric(d[col], errors='coerce').astype('Int64')

df = votos.merge(votacoes, left_on='votacao_id', right_on='id', suffixes=('', '_vt'))
df = df.merge(proposicoes, left_on='proposicao_id', right_on='id', suffixes=('', '_pr'))
df = df.merge(parlamentares, left_on='parlamentar_id', right_on='id', suffixes=('', '_pl'))

# Apenas Câmara, com tema definido e voto decisivo
df = df[df['casa'] == 'camara']
df = df[df['tema_cidadao'].notna()]
df = df[df['voto'].isin(['favoravel', 'contrario'])].copy()

df['fav'] = (df['voto'] == 'favoravel').astype(int)

print(f'\nVotos decisivos com tema: {len(df)}')
print(f'Parlamentares distintos: {df["parlamentar_id"].nunique()}')
print(f'Temas distintos: {df["tema_cidadao"].nunique()}')

2026-06-26 00:07:11,859 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=0&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:12,370 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=1000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:12,670 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=2000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:13,182 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=3000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:13,494 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/votos?select=votacao_id%2Cparlamentar_id%2Cvoto&offset=4000&limit=1000 "HTTP/2 200 OK"


...

2026-06-26 00:07:32,177 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=20000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:32,355 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=21000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:32,636 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/proposicoes?select=id%2Ctema_cidadao&offset=22000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:32,890 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/parlamentares?select=id%2Cnome%2Ccasa&offset=0&limit=1000 "HTTP/2 200 OK"


votos=55570  votacoes=137  proposicoes=22106  parlamentares=726

Votos decisivos com tema: 55233
Parlamentares distintos: 580
Temas distintos: 6


## 2. Agregação por parlamentar × tema

Para cada par (parlamentar, tema): `total_votacoes` = nº de votos decisivos e
`pct_favoravel` = % de favoráveis. Guardamos só pares com **≥ 3 votos** (evita
percentuais ruidosos de 1–2 votos). A postura segue os cortes do schema:
≥ 65% favorável · ≤ 35% contrário · senão neutro.

In [3]:
MIN_VOTOS = 1  # baixado de 3 para incluir temas com poucas votações (ex: Legislação Penal, Créditos)

agg = (
    df.groupby(['parlamentar_id', 'tema_cidadao'])
      .agg(total_votacoes=('fav', 'size'), favoraveis=('fav', 'sum'))
      .reset_index()
)
agg = agg[agg['total_votacoes'] >= MIN_VOTOS].copy()
agg['pct_favoravel'] = (agg['favoraveis'] / agg['total_votacoes'] * 100).round(2)

def classificar(pct):
    if pct >= 65:
        return 'favoravel'
    if pct <= 35:
        return 'contrario'
    return 'neutro'

agg['postura_geral'] = agg['pct_favoravel'].map(classificar)

print(f'Perfis (parlamentar × tema) com >= {MIN_VOTOS} votos: {len(agg)}')
print('\nDistribuição por postura_geral:')
print(agg['postura_geral'].value_counts())
print('\nTemas gerados:')
print(agg['tema_cidadao'].value_counts())
print('\nAmostra:')
print(agg.sort_values('total_votacoes', ascending=False).head(10).to_string(index=False))

Perfis (parlamentar × tema) com >= 1 votos: 3040

Distribuição por postura_geral:
postura_geral
favoravel    1546
neutro        964
contrario     530
Name: count, dtype: int64

Temas gerados:
tema_cidadao
Políticas Públicas e Programas Sociais     580
Educação, Trânsito e Direitos Sociais      562
Requerimentos e Regimento Interno          554
Legislação Penal e Proteção de Crianças    552
Sistema Único de Saúde (SUS)               502
Créditos e Orçamento Federal               290
Name: count, dtype: int64

Amostra:
 parlamentar_id                           tema_cidadao  total_votacoes  favoraveis  pct_favoravel postura_geral
             70 Políticas Públicas e Programas Sociais             107          70          65.42     favoravel
            582 Políticas Públicas e Programas Sociais             107          60          56.07        neutro
            376 Políticas Públicas e Programas Sociais             107          70          65.42     favoravel
            311 Políticas Púb

## 3. Gravar em `perfil_parlamentar` e validar

`upsert_perfil` faz upsert por (`parlamentar_id`, `tema_cidadao`), então reexecutar o
notebook atualiza os perfis sem duplicar.

In [4]:
registros = [
    {
        'parlamentar_id': int(r['parlamentar_id']),
        'tema_cidadao': r['tema_cidadao'],
        'pct_favoravel': float(r['pct_favoravel']),
        'total_votacoes': int(r['total_votacoes']),
        'postura_geral': r['postura_geral'],
    }
    for _, r in agg.iterrows()
]
total = upsert_perfil(registros)
print(f'{total} perfis gravados em perfil_parlamentar.')

2026-06-26 00:07:33,764 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22total_votacoes%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22postura_geral%22 "HTTP/2 200 OK"


2026-06-26 00:07:34,236 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22total_votacoes%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22postura_geral%22 "HTTP/2 200 OK"


2026-06-26 00:07:34,591 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22total_votacoes%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22postura_geral%22 "HTTP/2 200 OK"


2026-06-26 00:07:34,856 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22total_votacoes%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22postura_geral%22 "HTTP/2 200 OK"


2026-06-26 00:07:35,203 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22total_votacoes%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22postura_geral%22 "HTTP/2 200 OK"


2026-06-26 00:07:35,472 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22total_votacoes%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22postura_geral%22 "HTTP/2 200 OK"


2026-06-26 00:07:35,712 [INFO] HTTP Request: POST https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?on_conflict=parlamentar_id%2Ctema_cidadao&columns=%22parlamentar_id%22%2C%22total_votacoes%22%2C%22pct_favoravel%22%2C%22tema_cidadao%22%2C%22postura_geral%22 "HTTP/2 200 OK"


2026-06-26 00:07:35,715 [INFO] upsert perfil_parlamentar: 3040 registros


3040 perfis gravados em perfil_parlamentar.


In [5]:
# Sanidade: relê o que foi gravado
check = pd.DataFrame(buscar_todos(
    'perfil_parlamentar',
    'parlamentar_id,tema_cidadao,pct_favoravel,total_votacoes,postura_geral',
))
print(f'Perfis em perfil_parlamentar: {len(check)}')

if check.empty:
    print('\n⚠️  Tabela vazia. Causa provável: as proposições ligadas às votações ainda '
          'não têm `tema_cidadao` (rode o notebook 03 / Sprint 2 e grave os temas). '
          'Veja o print "Votos decisivos com tema" da célula 1: se for 0, é isso.')
else:
    print('\nDistribuição por postura_geral:')
    print(check['postura_geral'].value_counts())
    print('\nAmostra:')
    print(check.head(10).to_string(index=False))

2026-06-26 00:07:36,026 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=0&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:36,322 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=1000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:36,497 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=2000&limit=1000 "HTTP/2 200 OK"


2026-06-26 00:07:36,737 [INFO] HTTP Request: GET https://woxluznsfqskopzzjqtf.supabase.co/rest/v1/perfil_parlamentar?select=parlamentar_id%2Ctema_cidadao%2Cpct_favoravel%2Ctotal_votacoes%2Cpostura_geral&offset=3000&limit=1000 "HTTP/2 200 OK"


Perfis em perfil_parlamentar: 3040

Distribuição por postura_geral:
postura_geral
favoravel    1546
neutro        964
contrario     530
Name: count, dtype: int64

Amostra:
 parlamentar_id                      tema_cidadao  pct_favoravel  total_votacoes postura_geral
             10      Sistema Único de Saúde (SUS)           50.0               4        neutro
              6 Requerimentos e Regimento Interno          100.0               6     favoravel
              6      Sistema Único de Saúde (SUS)          100.0               3     favoravel
              7 Requerimentos e Regimento Interno          100.0               7     favoravel
              7      Sistema Único de Saúde (SUS)           75.0               4     favoravel
             54 Requerimentos e Regimento Interno          100.0               2     favoravel
            106 Requerimentos e Regimento Interno           75.0               8     favoravel
            106      Sistema Único de Saúde (SUS)           75.0    

In [6]:
print('votações c/ proposicao_id :', votacoes['proposicao_id'].notna().sum(), 'de', len(votacoes))
print('proposições c/ tema_cidadao:', proposicoes['tema_cidadao'].notna().sum(), 'de', len(proposicoes))

votações c/ proposicao_id : 137 de 137
proposições c/ tema_cidadao: 22105 de 22106
